# Chapter 2: Working with Text Data

Reference: `LLMs-from-scratch/ch02/`

**Goals:**
- Implement a BPE-like tokenizer
- Create an embedding layer
- Build a data loader with sliding window
- Understand token IDs, embeddings, and positional encodings

In [1]:
import tiktoken
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader

## 1. Tokenizer

In [2]:
#gpt-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, how are you doing today?"
id = tokenizer.encode(text)
decoded_text = tokenizer.decode(id)

print("Original text:", text)
print("Token IDs:", id)
print("Decoded text:", decoded_text)
print("Number of tokens:", len(id))

#individual tokens
tokens = [tokenizer.decode([token_id]) for token_id in id]
print(f"Individual tokens: {tokens}")

Original text: Hello, how are you doing today?
Token IDs: [15496, 11, 703, 389, 345, 1804, 1909, 30]
Decoded text: Hello, how are you doing today?
Number of tokens: 8
Individual tokens: ['Hello', ',', ' how', ' are', ' you', ' doing', ' today', '?']


In [3]:
text = "The quick brown fox jumps over the lazy dog."
id = tokenizer.encode(text)
print("Original text:", text)
print("Token IDs:", id)

#see the individual tokens
tokens = [tokenizer.decode([token_id]) for token_id in id]
print(f"Individual tokens: {tokens}")

Original text: The quick brown fox jumps over the lazy dog.
Token IDs: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]
Individual tokens: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']


## 2. Embeddings

In [4]:
vocab_size = 50257 #gpt-2 vocab size
embedding_dim = 768 #gpt-2 small embedding dimension

embedding_layer = nn.Embedding(vocab_size, embedding_dim) #create an embedding layer

#Tokenize and embed a short sentence
tokenizer = tiktoken.get_encoding("gpt2")
ids = tokenizer.encode("Hello world")
ids_tensor = torch.tensor(ids)

print("Token IDs:", ids_tensor)
print("shape:",(ids_tensor).shape)

embedded = embedding_layer(ids_tensor)
print("Embedded shape:", embedded.shape)
print("First token embedding: first 5 values:", embedded[0][:5])

Token IDs: tensor([15496,   995])
shape: torch.Size([2])
Embedded shape: torch.Size([2, 768])
First token embedding: first 5 values: tensor([ 1.1831, -0.7998, -0.8514,  0.6805,  0.5410], grad_fn=<SliceBackward0>)


## 3. Data Loader (Sliding Window)

In [5]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# Test on short text
txt = "The quick brown fox jumps over the lazy dog. " * 3
dataloader = create_dataloader_v1(txt, batch_size=2, max_length=8, stride=4)
for x, y in dataloader:
    print("Input IDs:", x)
    print("Target IDs:", y)
    print("Input shape:", x.shape, "| Target shape:", y.shape)
    break

Input IDs: tensor([[18045,   625,   262, 16931,  3290,    13,   383,  2068],
        [  383,  2068,  7586, 21831, 18045,   625,   262, 16931]])
Target IDs: tensor([[  625,   262, 16931,  3290,    13,   383,  2068,  7586],
        [ 2068,  7586, 21831, 18045,   625,   262, 16931,  3290]])
Input shape: torch.Size([2, 8]) | Target shape: torch.Size([2, 8])


## 4. Positional Encodings

In [6]:
#Create positional encoding layer

context_length = 1023 #small context length for testing
embedding_dim = 768 #gpt-2 small embedding dimension

post_embedding_layer = nn.Embedding(context_length, embedding_dim)

#Generate positional encodings for a sequence of length 8
pos_indices = torch.arange(8) #positions 0 to 7
print("Position indices:", pos_indices)

pos_embeddings = post_embedding_layer(pos_indices)
print("Positional embeddings shape:", pos_embeddings.shape)

#final input to the model is the sum of token embeddings and positional embeddings
token_ids = torch.tensor([15496, 995]) #token IDs for "Hello" and "world"
token_embeddings = embedding_layer(token_ids)
pos_embeddings = post_embedding_layer(pos_indices[:2]) #get positional embeddings for the first 2 tokens
input_embeddings = token_embeddings + pos_embeddings
print("Input embeddings shape:", input_embeddings.shape)
print("position embeds shape:", pos_embeddings.shape)
print("final input embeddings shape:", input_embeddings.shape)


Position indices: tensor([0, 1, 2, 3, 4, 5, 6, 7])
Positional embeddings shape: torch.Size([8, 768])
Input embeddings shape: torch.Size([2, 768])
position embeds shape: torch.Size([2, 768])
final input embeddings shape: torch.Size([2, 768])
